In [ ]:
!pip install pandas nltk contractions

import re
import string
import pandas as pd
import nltk
import contractions

# Download required NLTK corpuses
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

In [ ]:
# Replace 'IMDB Dataset.csv' with your local Kaggle dataset path
try:
    df = pd.read_csv('IMDB Dataset.csv')
except FileNotFoundError:
    # Fallback dummy dataset to verify the pipeline immediately
    df = pd.DataFrame({
        'review': [
            "I haven't seen a movie this good in years! Check http://movies.com <br /><br /> Loved it!",
            "Worst acting EVER... wouldn't recommend it to anyone. Completely wasted 2 hours."
        ],
        'sentiment': ['positive', 'negative']
    })

# Work with a small sample for rapid iteration
sample_df = df.head(5).copy()
print(sample_df[['review']].head())

In [ ]:
# Initialize tools
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Helper function to map NLTK POS tags to WordNet POS tags for better lemmatization accuracy
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

# 1. Expand Contractions
def expand_contractions(text):
    return contractions.fix(text)

# 2. Clean Text (HTML tags, URLs, lowercase)
def clean_text(text):
    text = re.sub(r'<.*?>', ' ', text)                  # Strip HTML tags like <br />
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)  # Strip URLs
    text = text.lower()                                  # Lowercasing
    return text

# 3. Remove Punctuation & Special Characters
def remove_punctuation(text):
    # Retain only alphanumeric characters and whitespace
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()            # Remove redundant whitespace
    return text

# 4. Tokenization
def tokenize_text(text):
    return word_tokenize(text)

# 5. Stop-word Removal
def remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words]

# 6a. Stemming
def apply_stemming(tokens):
    return [stemmer.stem(token) for token in tokens]

# 6b. Lemmatization (POS-informed)
def apply_lemmatization(tokens):
    pos_tags = nltk.pos_tag(tokens)
    return [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags]

In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
# Apply the pipeline step-by-step to capture state transitions
sample_df['expanded'] = sample_df['review'].apply(expand_contractions)
sample_df['cleaned'] = sample_df['expanded'].apply(clean_text)
sample_df['no_punct'] = sample_df['cleaned'].apply(remove_punctuation)
sample_df['tokens'] = sample_df['no_punct'].apply(tokenize_text)
sample_df['no_stopwords'] = sample_df['tokens'].apply(remove_stopwords)
sample_df['stemmed'] = sample_df['no_stopwords'].apply(apply_stemming)
sample_df['lemmatized'] = sample_df['no_stopwords'].apply(apply_lemmatization)

# Display before-and-after transformations for the first review
print("--- RAW TEXT ---")
print(sample_df['review'].iloc[0])

print("\n--- 1. EXPANDED CONTRACTIONS ---")
print(sample_df['expanded'].iloc[0])

print("\n--- 2 & 3. CLEANED & NO PUNCTUATION ---")
print(sample_df['no_punct'].iloc[0])

print("\n--- 4. TOKENIZED ---")
print(sample_df['tokens'].iloc[0][:15], "...")

print("\n--- 5. STOP-WORDS REMOVED ---")
print(sample_df['no_stopwords'].iloc[0][:15], "...")

print("\n--- 6a. STEMMED (Porter) ---")
print(sample_df['stemmed'].iloc[0][:15], "...")

print("\n--- 6b. LEMMATIZED (WordNet) ---")
print(sample_df['lemmatized'].iloc[0][:15], "...")

In [ ]:
def preprocess_pipeline(text, method='lemmatize'):
    """
    Consolidated end-to-end preprocessing function.
    method: 'lemmatize', 'stem', or 'both'
    """
    text = contractions.fix(text)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    tokens = word_tokenize(text)
    filtered_tokens = [w for w in tokens if w not in stop_words]
    
    if method == 'stem':
        return [stemmer.stem(w) for w in filtered_tokens]
    elif method == 'lemmatize':
        pos_tags = nltk.pos_tag(filtered_tokens)
        return [lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in pos_tags]
    elif method == 'both':
        pos_tags = nltk.pos_tag(filtered_tokens)
        lemmas = [lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in pos_tags]
        stems = [stemmer.stem(w) for w in filtered_tokens]
        return lemmas, stems

# Run across the whole dataset
# df['processed_tokens'] = df['review'].apply(preprocess_pipeline)

In [ ]:
len(df)

In [ ]:
# Check total size
print(f"Total available rows in df: {len(df)}")

# Automatically pick 5000 or the maximum rows available
sample_size = min(5000, len(df))
df_subset = df.sample(n=sample_size, random_state=42).copy()

print(f"Sampling {sample_size} rows for preprocessing...")
df_subset['lemmatized_tokens'] = df_subset['review'].apply(lambda x: preprocess_pipeline(x, method='lemmatize'))
df_subset['clean_text'] = df_subset['lemmatized_tokens'].apply(lambda tokens: " ".join(tokens))
print("Preprocessing complete!")

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Join tokens back into a single string for vectorization
# (If you ran the steps on df instead of sample_df, use df['lemmatized'])
df['clean_text'] = df['lemmatized'].apply(lambda tokens: " ".join(tokens))

# 2. Encode target labels ('positive' -> 1, 'negative' -> 0)
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# 3. Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], 
    df['label'], 
    test_size=0.20, 
    random_state=42, 
    stratify=df['label']
)

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")